# Train model

In [1]:
import graphein
import lightning as L
import pandas as pd
import torch
import torch.nn.functional as F
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from optimizer.sam import SAM
from torch import nn
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import Compose
import shutil  # Only used to remove processed dir
# This might have an error when coming back to Boston
from graphein.ml.datasets import (
    ProteinGraphDataset,
    ProteinGraphListDataset,
)
from graphein.ml.conversion import GraphFormatConvertor 

from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_distance_to_edges,
    add_hydrogen_bond_interactions,
    add_ionic_interactions,
    add_peptide_bonds,
)
from pathlib import Path

from utils.features import DatasetNodeAnnotator, DatasetEdgeAnnotator

/home/bobby/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Set constants

In [2]:
# Training and validation dataset paths to train and stop models
training_data_samplesheet_path = "data/dummy_samplesheet.csv"

SEED = 255
L.seed_everything(SEED)

Seed set to 255


255

In [3]:
EDGE_CONSTRUCTION_FNCS = [
    add_peptide_bonds,
    add_ionic_interactions,
    add_hydrogen_bond_interactions,
    add_distance_to_edges,
]

config = ProteinGraphConfig(edge_construction_functions=EDGE_CONSTRUCTION_FNCS)
config.model_dump()

{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [<function graphein.protein.edges.distance.add_peptide_bonds(G: 'nx.Graph') -> 'nx.Graph'>,
  <function graphein.protein.edges.distance.add_ionic_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_hydrogen_bond_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_distance_to_edges(G: 'nx.Graph') -> 'nx.Graph'>],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.meiler_embedding(n: str, d: Dict[str, Any], return_array: bool = False) -> Union[pandas.core.series.Series, numpy.ndarray]>],
 'edge_metadata_functions': None,
 'graph_metadata_functions': None,
 'get_contacts_conf

# Load raw data and untrained model

## Training data

In [4]:
samplesheet = pd.read_csv(training_data_samplesheet_path)

pdb_paths = samplesheet["structure_path"].tolist()
affinity_values = [torch.tensor(dg) for dg in samplesheet["dg"]]

In [5]:
dataset_convertor = GraphFormatConvertor(
    src_format="nx",
    dst_format="pyg",
    # verbose="all_info",
    columns=[
        "edge_index",
        "coords",
        "node_id",
        "meiler",  # Meiler embeddings
        "kind",  # Type of bond (hydrogen, ionic, etc.)
        "dist_mat",
        "distance",
    ],
)

In [6]:
node_functions = ["meiler_embedding"]
edge_functions = ["distance_to_edges"]

node_annotator = DatasetNodeAnnotator(node_functions)
edge_annotator = DatasetEdgeAnnotator(edge_functions)

In [7]:
processed_dir = Path("processed")
if processed_dir.exists():
    shutil.rmtree(processed_dir)

# Regular ProteinGraphDataset
ds = ProteinGraphDataset(
    root=".",
    paths=pdb_paths,
    graph_labels=affinity_values,
    graph_format_convertor=dataset_convertor,
    graphein_config=config,
    pre_transform=Compose(
        [node_annotator.set_node_x, edge_annotator.set_edge_attr]
    ),  # Adds meiler embeddings as x value and edge distance as edge_attr
)

Processing...
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.18it/s]
Done!


## Define and init model class

In [8]:
class DenseLayer(nn.Module):
    """Convinience layer for combinding linear, leaky_relu, and
    dropout into one Module
    """

    def __init__(
        self,
        in_layers,
        out_layers,
        negative_slope=0.1,
        p=0.3,
    ):
        super().__init__()

        self.layer = nn.Sequential(
            nn.Linear(in_layers, out_layers),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p),
        )

    def forward(self, x):
        return self.layer(x)

In [9]:
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool, global_add_pool

In [10]:
# global_mean_pool, global_max_pool, global_add_pool
# Use this!!!! CGConv. Just not in the beginning

# HeteroConv and/or GINEConv??

In [11]:
help(GCNConv)

Help on class GCNConv in module torch_geometric.nn.conv.gcn_conv:

class GCNConv(torch_geometric.nn.conv.message_passing.MessagePassing)
 |  GCNConv(in_channels: int, out_channels: int, improved: bool = False, cached: bool = False, add_self_loops: Optional[bool] = None, normalize: bool = True, bias: bool = True, **kwargs)
 |
 |  The graph convolutional operator from the `"Semi-supervised
 |  Classification with Graph Convolutional Networks"
 |  <https://arxiv.org/abs/1609.02907>`_ paper.
 |
 |  .. math::
 |      \mathbf{X}^{\prime} = \mathbf{\hat{D}}^{-1/2} \mathbf{\hat{A}}
 |      \mathbf{\hat{D}}^{-1/2} \mathbf{X} \mathbf{\Theta},
 |
 |  where :math:`\mathbf{\hat{A}} = \mathbf{A} + \mathbf{I}` denotes the
 |  adjacency matrix with inserted self-loops and
 |  :math:`\hat{D}_{ii} = \sum_{j=0} \hat{A}_{ij}` its diagonal degree matrix.
 |  The adjacency matrix can include other values than :obj:`1` representing
 |  edge weights via the optional :obj:`edge_weight` tensor.
 |
 |  Its node-

In [12]:
class RBFExpansion(nn.Module):
    def __init__(self, d_min=0.0, d_max=10.0, num_rbf=16, gamma=10.0):
        super().__init__()
        centers = torch.linspace(d_min, d_max, num_rbf)
        self.register_buffer('centers', centers)
        self.gamma = gamma

    def forward(self, distances):
        # distances: [num_edges, 1]
        diff = distances - self.centers.view(1, -1)  # [num_edges, num_rbf]
        return torch.exp(-self.gamma * diff.pow(2))

In [13]:
from typing import Callable

In [14]:
class GINEMLP(nn.Module):
    """ADD BATCH NORM AND DROPOUT!!!"""
    def __init__(self, hidden_layer_size: int, layer_count: int = 1, act_fnc: Callable = nn.LeakyReLU, act_fnc_kwargs: dict = {}):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.layer_count = layer_count
        self.act_fnc = act_fnc
        self.act_fnc_kwargs = act_fnc_kwargs

    @property
    def mlp(self):
        mlp = nn.ModuleList()
        for _ in range(self.layer_count):
            layer = nn.Sequential(
                nn.Linear(self.hidden_layer_size, self.hidden_layer_size), 
                self.act_fnc(**self.act_fnc_kwargs)
            )
            mlp.append(layer)
        mlp.append(nn.Linear(self.hidden_layer_size, self.hidden_layer_size))
        
        return mlp

    def forward(self, x):
        for layer in self.mlp:
            x = layer(x)
        return x

In [82]:
x = GINEMLP(hidden_layer_size=5, layer_count=2)
x.eval()

x(torch.Tensor(5))

tensor([-0.5889, -0.0681, -0.2620,  0.3476,  0.1473], grad_fn=<ViewBackward0>)

In [15]:
class BaseAbAffinityPredictionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Convokutional layers
        self.conv1 = GCNConv(5, 10)
        self.conv2 = GCNConv(10, 20)
        self.rbf = RBFExpansion()

        # Regression head
        self.regression_head = nn.Sequential(
            DenseLayer(10, 5),
            DenseLayer(5, 3),
            nn.Linear(3, 1),
        )

    def forward(self, x, edge_index):

        # X conv chain
        # x = global_max_pool(x, edge_index)
        y_pred = self.regression_head(x)
        return y_pred

In [16]:
model = BaseAbAffinityPredictionModel()
model.eval()

BaseAbAffinityPredictionModel(
  (conv1): GCNConv(5, 10)
  (conv2): GCNConv(10, 20)
  (regression_head): Sequential(
    (0): DenseLayer(
      (layer): Sequential(
        (0): Linear(in_features=10, out_features=5, bias=True)
        (1): LeakyReLU(negative_slope=0.1)
        (2): Dropout(p=0.3, inplace=False)
      )
    )
    (1): DenseLayer(
      (layer): Sequential(
        (0): Linear(in_features=5, out_features=3, bias=True)
        (1): LeakyReLU(negative_slope=0.1)
        (2): Dropout(p=0.3, inplace=False)
      )
    )
    (2): Linear(in_features=3, out_features=1, bias=True)
  )
)

In [17]:
# x = torch.Tensor(range(10))
# model(x)

In [18]:
# class LitAutoEncoder(L.LightningModule):
#     def __init__(self, encoder, decoder):
#         super().__init__()
#         self.encoder = encoder
#         self.decoder = decoder

#     def training_step(self, batch, batch_idx):
#         # training_step defines the train loop.
#         x, _ = batch
#         x = x.view(x.size(0), -1)
#         z = self.encoder(x)
#         x_hat = self.decoder(z)
#         loss = F.mse_loss(x_hat, x)
#         return loss

#     def configure_optimizers(self):
#         optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
#         return optimizer

In [22]:
class AbAffinityPredictionModel(L.LightningModule):
    def __init__(self, affinity_model: BaseAbAffinityPredictionModel):
        super().__init__()
        self.model = affinity_model

    def forward(self, x, edge_index):
        return self.model(x, edge_index)

    def _generate_loss(self, batch, batch_idx):
        x, y = batch.x, batch.graph_y
        y_pred = self.affinity_model(x)
        loss = F.mse_loss(y_pred, y)
        return loss

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss = self._generate_loss(batch, batch_idx)
        return loss

    def validation_step(self, batch, batch_idx):
        # this is the validation loop
        val_loss = self._generate_loss(batch, batch_idx)
        self.log("val_loss", val_loss)

    def test_step(self, batch, batch_idx):
        # training_step defines the train loop.
        test_loss = self._generate_loss(batch, batch_idx)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        # optimizer = SAM()
        return optimizer

In [23]:
lightning_model = AbAffinityPredictionModel(model)

In [ ]:
# lightning_model(x)

# Preprocess data

In [ ]:
# Define X

In [ ]:
trainer = L.Trainer()
help(trainer.test)

In [ ]:
# DataLoader

In [ ]:
# x_train --> x_train, x_val

# train_set_size = int(len(train_set) * 0.8)
# valid_set_size = len(train_set) - train_set_size

# split the train set into two
# seed = torch.Generator().manual_seed(SEED)
# train_set, valid_set = data.random_split(train_set, [train_set_size, valid_set_size], generator=seed)
# train_loader, valid_loader)

In [ ]:
# train_loader = DataLoader
# val data_loader = DataLoader

# Train model

In [ ]:
# # model
# autoencoder = LitAutoEncoder(Encoder(), Decoder())

# # train model
# trainer = L.Trainer()
# trainer.fit(model=autoencoder, train_dataloaders=train_loader)

In [ ]:
# callbacks=[EarlyStopping(monitor="val_loss", mode="min")]

In [ ]:
# Trainer
trainer = L.Trainer()
# trainer.fit(model, train_loader, valid_loader)

In [ ]:
# Once the model has finished training, call .test
# trainer.test(model, dataloaders=DataLoader(test_set))